<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>AI SYSTEMS ENGINEERING 2</b>

## Unit 5: Research Assistant with MCP

**CV Raman Global University, Bhubaneswar**
*AI Center of Excellence*

</div>

---

### What You'll Learn

In this notebook, you will:

1. **Build an MCP client** — use `stdio_client` and `ClientSession` to interact with servers directly
2. **Create specialized MCP servers** — knowledge base, notes, and dictionary servers
3. **Assemble a multi-server research assistant** — connect all servers to one agent
4. **Test with real queries** — research topics, take notes, define terms
5. **Add a Gradio interface** — wrap the assistant in a chat UI

## 1. Environment Setup

In [ ]:
!pip install -q openai-agents mcp gradio

In [ ]:
import os
import getpass
import json

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key: ")

In [ ]:
# Colab/Jupyter compatibility for MCP stdio transport
import sys, io, os

try:
    sys.stderr.fileno()
except (io.UnsupportedOperation, AttributeError):
    sys.stderr = open(os.devnull, "w")

---

## 2. Building an MCP Client

In the previous notebook, we used `MCPServerStdio` from the OpenAI Agents SDK, which handles the client connection for us. But you can also build **standalone MCP clients** using the `mcp` library directly.

This is useful when you want to:
- Call tools programmatically (without an agent)
- Read resources from a server
- Inspect server capabilities in detail

In [ ]:
# First, create a simple test server
sample_server_code = '''
from mcp.server.fastmcp import FastMCP
import json

mcp = FastMCP("sample_server")

data_store = {
    "items": ["apple", "banana", "cherry"],
    "count": 3
}

@mcp.tool()
async def add_item(item: str) -> str:
    """Add an item to the list.
    
    Args:
        item: The item to add
    """
    data_store["items"].append(item)
    data_store["count"] += 1
    return f"Added {item}. Total items: {data_store[\"count\"]}"

@mcp.tool()
async def get_items() -> str:
    """Get all items in the list."""
    return json.dumps(data_store["items"])

@mcp.resource("data://items")
async def items_resource() -> str:
    """Get all items as a resource."""
    return json.dumps(data_store)

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open("sample_server.py", "w") as f:
    f.write(sample_server_code)

print("Created sample_server.py")
print("Tools: add_item, get_items")
print("Resources: data://items")

In [ ]:
import mcp
from mcp.client.stdio import stdio_client
from mcp import StdioServerParameters

server_params = StdioServerParameters(
    command="python",
    args=["sample_server.py"],
    env=None
)

async def client_demo():
    """Demonstrate direct MCP client usage."""
    
    try:
        async with stdio_client(server_params, errlog=open(os.devnull, "w")) as streams:
            async with mcp.ClientSession(*streams, read_timeout_seconds=30) as session:
                await session.initialize()
                print("Connected to MCP server!\n")
                
                # 1. List available tools
                tools_result = await session.list_tools()
                print("AVAILABLE TOOLS")
                print("=" * 50)
                for tool in tools_result.tools:
                    print(f"  - {tool.name}: {tool.description}")
                
                # 2. Call tools programmatically
                print("\nCALLING TOOLS")
                print("=" * 50)
                
                result = await session.call_tool("get_items", {})
                print(f"  get_items(): {result.content[0].text}")
                
                result = await session.call_tool("add_item", {"item": "orange"})
                print(f"  add_item('orange'): {result.content[0].text}")
                
                result = await session.call_tool("get_items", {})
                print(f"  get_items(): {result.content[0].text}")
                
                # 3. Read a resource
                print("\nREADING RESOURCE")
                print("=" * 50)
                result = await session.read_resource("data://items")
                print(f"  data://items: {result.contents[0].text}")
    except ExceptionGroup:
        pass  # Suppress subprocess cleanup errors in Colab

await client_demo()

> **Key insight:** `stdio_client` + `ClientSession` gives you low-level access to MCP servers. You can list tools, call them with arguments, and read resources — all without an LLM agent. This is useful for testing servers and building custom integrations.

---

## 3. Building MCP Servers

Now let's build three specialized MCP servers for our research assistant:

1. **Knowledge Base** — search articles and get full content
2. **Notes** — create and list research notes
3. **Dictionary** — define terms and look up facts

### Server 1: Knowledge Base

In [ ]:
knowledge_server_code = '''
from mcp.server.fastmcp import FastMCP
import json

mcp = FastMCP("knowledge_server")

knowledge_base = {
    "artificial_intelligence": {
        "title": "Artificial Intelligence",
        "summary": "AI is the simulation of human intelligence by machines.",
        "content": """Artificial Intelligence (AI) refers to the simulation of human intelligence 
in machines. The term was coined in 1956 at the Dartmouth Conference. AI systems can perform 
tasks that typically require human intelligence, such as visual perception, speech recognition, 
decision-making, and language translation. Modern AI is powered by machine learning, especially 
deep learning with neural networks.""",
        "tags": ["technology", "machine learning", "computing"]
    },
    "machine_learning": {
        "title": "Machine Learning",
        "summary": "ML is a subset of AI that enables systems to learn from data.",
        "content": """Machine Learning (ML) is a subset of artificial intelligence that provides 
systems the ability to automatically learn and improve from experience. The key types are:
1. Supervised Learning: Learning from labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through trial and error""",
        "tags": ["AI", "data science", "algorithms"]
    },
    "model_context_protocol": {
        "title": "Model Context Protocol (MCP)",
        "summary": "MCP is an open standard for connecting AI agents to external tools.",
        "content": """The Model Context Protocol (MCP) is an open standard developed by Anthropic 
for connecting AI agents to external data sources and tools. MCP Architecture:
- Host: The application containing the AI agent
- Client: Connects hosts to servers
- Server: Provides tools, resources, and prompts
Benefits: Write tools once, use everywhere. Automatic tool discovery.""",
        "tags": ["AI", "protocol", "tools", "agents"]
    }
}

@mcp.tool()
async def search_knowledge(query: str) -> str:
    """Search the knowledge base for information on a topic.
    
    Args:
        query: The search query (topic, keyword, or phrase)
    """
    results = []
    query_lower = query.lower()
    
    for key, article in knowledge_base.items():
        if (query_lower in article["title"].lower() or 
            query_lower in article["summary"].lower() or
            query_lower in article["content"].lower()):
            results.append({"id": key, "title": article["title"], "summary": article["summary"]})
    
    if results:
        return json.dumps({"found": len(results), "results": results}, indent=2)
    return f"No articles found matching: {query}"

@mcp.tool()
async def get_article(article_id: str) -> str:
    """Get the full content of a specific article.
    
    Args:
        article_id: The ID of the article
    """
    if article_id in knowledge_base:
        article = knowledge_base[article_id]
        return f"# {article[\"title\"]}\\n\\n{article[\"content\"]}"
    return f"Article not found: {article_id}"

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open("research_knowledge_server.py", "w") as f:
    f.write(knowledge_server_code)

print("Created research_knowledge_server.py")
print("Tools: search_knowledge, get_article")

### Server 2: Notes

In [ ]:
notes_server_code = '''
from mcp.server.fastmcp import FastMCP
from datetime import datetime
import json

mcp = FastMCP("research_notes_server")

notes = {}
note_counter = 0

@mcp.tool()
async def create_note(title: str, content: str, category: str = "general") -> str:
    """Create a new research note.
    
    Args:
        title: The title of the note
        content: The content/body of the note
        category: Category for organization
    """
    global note_counter
    note_counter += 1
    note_id = f"note_{note_counter}"
    
    notes[note_id] = {
        "id": note_id,
        "title": title,
        "content": content,
        "category": category,
        "created_at": datetime.now().strftime("%Y-%m-%d %H:%M")
    }
    
    return f"Note created: {note_id} - {title}"

@mcp.tool()
async def list_notes(category: str = "") -> str:
    """List all notes, optionally filtered by category.
    
    Args:
        category: Optional category filter
    """
    if not notes:
        return "No notes found."
    
    filtered = list(notes.values())
    if category:
        filtered = [n for n in filtered if n["category"].lower() == category.lower()]
    
    if not filtered:
        return f"No notes found in category: {category}"
    
    lines = ["Your Notes:"]
    for note in filtered:
        lines.append(f"  [{note[\"id\"]}] {note[\"title\"]} ({note[\"category\"]})")
    return "\\n".join(lines)

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open("research_notes_server.py", "w") as f:
    f.write(notes_server_code)

print("Created research_notes_server.py")
print("Tools: create_note, list_notes")

### Server 3: Dictionary

In [ ]:
dictionary_server_code = '''
from mcp.server.fastmcp import FastMCP
import json

mcp = FastMCP("dictionary_server")

definitions = {
    "algorithm": {
        "definition": "A step-by-step procedure for solving a problem.",
        "example": "A sorting algorithm arranges data in a specific order."
    },
    "neural network": {
        "definition": "A computing system inspired by biological neural networks.",
        "example": "Image recognition uses convolutional neural networks."
    },
    "agent": {
        "definition": "An autonomous entity that perceives its environment and takes actions.",
        "example": "A chatbot is an AI agent that converses with users."
    },
    "mcp": {
        "definition": "Model Context Protocol - an open standard for connecting AI agents to tools.",
        "example": "MCP servers provide tools that AI agents can use."
    }
}

facts = {
    "python": "Python was created by Guido van Rossum and released in 1991.",
    "ai": "The term \'Artificial Intelligence\' was coined at the Dartmouth Conference in 1956.",
    "mcp": "MCP was developed by Anthropic to standardize how AI agents connect to tools."
}

@mcp.tool()
async def define(term: str) -> str:
    """Get the definition of a technical term.
    
    Args:
        term: The term to define
    """
    term_lower = term.lower()
    if term_lower in definitions:
        entry = definitions[term_lower]
        return f"Definition of {term.upper()}:\\n{entry[\"definition\"]}\\n\\nExample: {entry[\"example\"]}"
    return f"Definition not found for: {term}"

@mcp.tool()
async def get_fact(topic: str) -> str:
    """Get a quick fact about a topic.
    
    Args:
        topic: The topic to get a fact about
    """
    topic_lower = topic.lower()
    if topic_lower in facts:
        return f"Fact about {topic}: {facts[topic_lower]}"
    return f"No fact found for: {topic}"

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open("research_dictionary_server.py", "w") as f:
    f.write(dictionary_server_code)

print("Created research_dictionary_server.py")
print("Tools: define, get_fact")

---

## 4. Testing the Servers

Let's verify all three servers start correctly and expose the expected tools:

In [ ]:
from agents import Agent, Runner
from agents.mcp import MCPServerStdio

knowledge_params = {"command": "python", "args": ["research_knowledge_server.py"]}
notes_params = {"command": "python", "args": ["research_notes_server.py"]}
dictionary_params = {"command": "python", "args": ["research_dictionary_server.py"]}

async def test_all_servers():
    print("TESTING MCP SERVERS")
    print("=" * 50)
    
    print("\n1. Testing Knowledge Server...")
    async with MCPServerStdio(params=knowledge_params, client_session_timeout_seconds=30) as server:
        tools = await server.list_tools()
        print(f"   Tools: {[t.name for t in tools]}")
    print("   Status: OK")
    
    print("\n2. Testing Notes Server...")
    async with MCPServerStdio(params=notes_params, client_session_timeout_seconds=30) as server:
        tools = await server.list_tools()
        print(f"   Tools: {[t.name for t in tools]}")
    print("   Status: OK")
    
    print("\n3. Testing Dictionary Server...")
    async with MCPServerStdio(params=dictionary_params, client_session_timeout_seconds=30) as server:
        tools = await server.list_tools()
        print(f"   Tools: {[t.name for t in tools]}")
    print("   Status: OK")
    
    print("\n" + "=" * 50)
    print("ALL SERVERS READY!")

await test_all_servers()

---

## 5. Building the Research Assistant

Now let's connect all three servers to a single agent. The agent's instructions tell it how to use each server's tools:

In [ ]:
ASSISTANT_INSTRUCTIONS = """
You are a helpful Research Assistant with access to multiple tools:

1. KNOWLEDGE BASE: search_knowledge, get_article
2. NOTES: create_note, list_notes
3. DICTIONARY: define, get_fact

When users ask about topics, search the knowledge base first.
Explain technical terms using the dictionary.
Offer to save important findings as notes.
"""

async def run_research_query(query: str):
    """Run a single research query."""
    
    async with MCPServerStdio(params=knowledge_params, client_session_timeout_seconds=30) as knowledge_server:
        async with MCPServerStdio(params=notes_params, client_session_timeout_seconds=30) as notes_server:
            async with MCPServerStdio(params=dictionary_params, client_session_timeout_seconds=30) as dictionary_server:
                
                agent = Agent(
                    name="ResearchAssistant",
                    instructions=ASSISTANT_INSTRUCTIONS,
                    mcp_servers=[knowledge_server, notes_server, dictionary_server],
                    model="gpt-4o-mini"
                )
                
                result = await Runner.run(agent, query)
                return result.final_output

print("Research Assistant function defined!")

> **Key insight:** The agent sees tools from ALL three servers and decides which ones to call based on the query. A single query might trigger tools from multiple servers — e.g., searching the knowledge base, then saving a summary as a note.

---

## 6. Testing the Research Assistant

In [ ]:
print("QUERY 1: Learn about a topic")
print("=" * 50)

response = await run_research_query(
    "What is MCP? Search the knowledge base and explain it to me."
)
print(f"\n{response}")

In [ ]:
print("QUERY 2: Research and take notes")
print("=" * 50)

response = await run_research_query(
    """Search for information about machine learning. 
    Create a summary note titled 'ML Overview' with the key points."""
)
print(f"\n{response}")

In [ ]:
print("QUERY 3: Define terms and get facts")
print("=" * 50)

response = await run_research_query(
    "Define 'algorithm' and 'neural network'. Also give me a fact about Python."
)
print(f"\n{response}")

---

## 7. Gradio Interface

Let's wrap our research assistant in a chat UI using Gradio:

In [ ]:
import gradio as gr
import asyncio

async def research_chat(message, history):
    try:
        response = await run_research_query(message)
        return response
    except Exception as e:
        return f"Error: {str(e)}"

def chat_wrapper(message, history):
    return asyncio.run(research_chat(message, history))

demo = gr.ChatInterface(
    fn=chat_wrapper,
    title="Research Assistant",
    description="""A research assistant powered by MCP servers.
    
I can: Search knowledge base, Define terms, Take notes, Get facts
    """,
    examples=[
        "Search for information about artificial intelligence",
        "Tell me about machine learning",
        "Define the term 'algorithm'",
        "Create a note titled 'Research Ideas' with my thoughts on AI"
    ],
    theme="soft"
)

print("Gradio interface created!")

In [ ]:
demo.launch(share=False)

---

## 8. Key Takeaways

| Concept | What it does |
|---------|-------------|
| **stdio_client + ClientSession** | Low-level MCP client for calling tools and reading resources directly |
| **Specialized servers** | Each server focuses on one domain (knowledge, notes, dictionary) |
| **Multi-server agent** | One agent connects to multiple servers, accessing all their tools |
| **Agent instructions** | Guide the LLM on which tools to use for different types of queries |
| **Gradio integration** | Wraps the agent in a user-friendly chat interface |

---

## 9. Exercises

### Exercise 1: Add a New Server
Create a "Bookmarks" MCP server with tools for `save_bookmark(url, title, tags)`, `list_bookmarks()`, and `search_bookmarks(query)`. Add it to the research assistant and test queries that combine bookmarks with knowledge search.

### Exercise 2: Enhanced Gradio UI
Modify the Gradio interface to show a sidebar with the list of saved notes and bookmarks. Add buttons for common actions like "List all notes" and "Show available topics".

---

**Congratulations!** You've completed Unit 5: Introduction to MCP

You've learned how to build MCP servers, connect agents to them, build standalone clients, and assemble a multi-server research assistant with a Gradio interface.

---

**Course Information:**
- **Institution:** CV Raman Global University, Bhubaneswar
- **Program:** AI Center of Excellence
- **Course:** AI Systems Engineering 2
- **Developed by:** [Poorit Technologies](https://poorit.in) - *Transform Graduates into Industry-Ready Professionals*

---